<a href="https://colab.research.google.com/github/Dubnitskyi/ai_kursova/blob/main/voice_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Голосовий помічник

Основна ідея проєкту полягає в тому, що користувач вимовляє ключову фразу, після чого система записує голосову команду, розпізнає її, виконує дію та озвучує відповідь.

У проєкті реалізовано такі основні функції:

- розпізнавання wake word;
- розпізнавання голосової команди;
- виконання простих команд;
- синтез голосової відповіді.

# 1. Налаштування Wake Word

In [ ]:
!pip install openwakeword vosk edge-tts onnxruntime numpy
!apt-get install -y ffmpeg

In [ ]:
import os
import json
import wave
import asyncio
import numpy as np

from base64 import b64decode
from IPython.display import Audio, display, Javascript
from google.colab import output, files

from openwakeword.model import Model
import vosk
import edge_tts

In [ ]:
from google.colab import output
from IPython.display import Javascript, display
from base64 import b64decode
import os

RECORD_JS = """
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const mediaRecorder = new MediaRecorder(stream);
  let chunks = [];

  mediaRecorder.ondataavailable = e => chunks.push(e.data);
  mediaRecorder.start();

  await new Promise(resolve => setTimeout(resolve, seconds * 1000));

  mediaRecorder.stop();

  await new Promise(resolve => mediaRecorder.onstop = resolve);

  stream.getTracks().forEach(track => track.stop());

  const blob = new Blob(chunks, { type: 'audio/webm' });
  const reader = new FileReader();

  return await new Promise(resolve => {
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}

window.recordAudio = recordAudio;
"""

display(Javascript(RECORD_JS))


def record_audio(seconds=3, filename="record.wav"):
    display(Javascript(RECORD_JS))

    print(f"Запис аудіо протягом {seconds} секунд...")

    audio_data = output.eval_js(f"window.recordAudio({seconds})")

    audio_bytes = b64decode(audio_data.split(",")[1])

    webm_path = "temp_audio.webm"

    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    os.system(f"ffmpeg -y -i {webm_path} -ar 16000 -ac 1 {filename} > /dev/null 2>&1")

    print(f"Аудіо збережено у файл: {filename}")

    return filename

## Запис аудіо з мікрофона

Google Colab не має прямого доступу до мікрофона.  
Тому для запису аудіо використовується JavaScript-код, який отримує доступ до мікрофона через браузер, записує звук і передає його назад.

Функція `record_audio()` записує звук із мікрофона користувача.

Спочатку JavaScript отримує дозвіл на використання мікрофона. Після цього аудіо записується у форматі `webm`.  
Оскільки бібліотека Vosk працює з аудіо у форматі WAV із частотою 16000 Гц і одним каналом, файл конвертується за допомогою `ffmpeg`.

У результаті функція повертає шлях до WAV-файлу, який далі використовується для розпізнавання wake word або голосової команди.

In [ ]:
n
!wget -q https://alphacephei.com/vosk/models/vosk-model-small-uk-v3-small.zip
!unzip -q vosk-model-small-uk-v3-small.zip

Завантаженння української Vosk моделі

In [ ]:
wake_model = Model()

print("Стандартна wake word модель успішно завантажена")
print("Для активації скажіть: Hey Jarvis")

## Завантаження wake word моделі

На цьому етапі завантажується власна модель wake word.

Модель була попередньо натренована за допомогою openWakeWord та збережена у форматі ONNX.

Основна задача даної моделі — визначити момент активації голосового помічника після вимовлення користувачем ключової фрази.

In [ ]:
def check_wake_word(audio_path, threshold=0.3):
    with wave.open(audio_path, "rb") as wf:
        audio = wf.readframes(wf.getnframes())
        audio = np.frombuffer(audio, dtype=np.int16)

    frame_size = 1280
    max_score = 0
    best_word = None

    for i in range(0, len(audio), frame_size):
        frame = audio[i:i + frame_size]

        if len(frame) < frame_size:
            break

        prediction = wake_model.predict(frame)

        current_word = max(prediction, key=prediction.get)
        current_score = prediction[current_word]

        if current_score > max_score:
            max_score = current_score
            best_word = current_word

    print(f"Найбільш ймовірне wake word: {best_word}")
    print(f"Максимальна оцінка wake word: {max_score:.4f}")

    return max_score >= threshold

## Реалізація перевірки wake word

Після запису аудіо необхідно визначити, чи була вимовлена ключова фраза.

Для цього аудіозапис розбивається на окремі фрагменти та передається до моделі openWakeWord.

Модель повертає оцінку від 0 до 1.

Чим ближче значення до 1, тим вища ймовірність правильної активації.

In [ ]:
wake_audio = record_audio(
    seconds=3,
    filename="wake.wav"
)

result = check_wake_word(
    audio_path=wake_audio,
    threshold=0.3
)

print("Wake word знайдено:", result)

## Тестування wake word

Перед переходом до розпізнавання команд необхідно перевірити працездатність моделі.

Для цього буде записано коротке аудіо тривалістю 3 секунди та виконано аналіз на наявність ключової фрази.

# 2. Розпізнавання голосових команд

In [ ]:
stt_model = vosk.Model("vosk-model-small-uk-v3-small")

In [ ]:
def recognize_speech(audio_path):
    wf = wave.open(audio_path, "rb")

    recognizer = vosk.KaldiRecognizer(stt_model, 16000)

    recognized_text = ""

    while True:
        data = wf.readframes(4000)

        if len(data) == 0:
            break

        if recognizer.AcceptWaveform(data):
            result = json.loads(recognizer.Result())
            recognized_text += result.get("text", "") + " "

    final_result = json.loads(recognizer.FinalResult())
    recognized_text += final_result.get("text", "")

    recognized_text = recognized_text.strip()

    print("Розпізнана команда:", recognized_text)

    return recognized_text

## Функція розпізнавання мовлення

Функція `recognize_speech()` отримує шлях до аудіофайлу, відкриває його та передає аудіодані до Vosk.

Результатом роботи функції є текст, який був розпізнаний із голосової команди користувача.

In [ ]:
command_audio = record_audio(
    seconds=5,
    filename="command.wav"
)

command_text = recognize_speech(command_audio)

## Тестування розпізнавання голосової команди

На цьому етапі записується короткий аудіофайл із голосовою командою користувача.

Для тестування можна сказати:

- відкрий гугл;
- відкрий ютуб;
- котра година;

# 3. Виконання команд

Після розпізнавання голосової команди текст передається до функції виконання команд.


In [ ]:
from IPython.display import HTML, display
from datetime import datetime
from zoneinfo import ZoneInfo

def execute_command(command):
    command = command.lower()

    if "гугл" in command or "google" in command or "браузер" in command:
        display(HTML('<a href="https://google.com" target="_blank">Відкрити Google</a>'))
        return "Команда виконана. Посилання на Google виведено нижче."

    elif "ютуб" in command or "youtube" in command:
        display(HTML('<a href="https://youtube.com" target="_blank">Відкрити YouTube</a>'))
        return "Команда виконана. Посилання на YouTube виведено нижче."

    elif "час" in command or "година" in command:
        current_time = datetime.now(ZoneInfo("Europe/Kyiv")).strftime("%H:%M")
        return f"Зараз {current_time}"

    elif "привіт" in command:
        return "Привіт. Я голосовий помічник Jarvis"

    else:
        return "Команду не розпізнано"

## Тестування виконання команди

Тепер перевіримо, яку відповідь повертає система після розпізнавання голосової команди.

In [ ]:
response = execute_command(command_text)

print("Відповідь системи:", response)

# 4. Синтез голосової відповіді

Останнім етапом роботи голосового помічника є озвучення відповіді.

Для цього використовується бібліотека `edge-tts`, яка дозволяє перетворювати текст у мовлення.

У проєкті використовується український голос `uk-UA-PolinaNeural`.

In [ ]:
async def speak(text, filename="response.mp3"):
    voice = "uk-UA-PolinaNeural"

    communicate = edge_tts.Communicate(
        text=text,
        voice=voice
    )

    await communicate.save(filename)

    display(Audio(filename, autoplay=True))

In [ ]:
await speak(response)

# 5. Повний сценарій роботи голосового помічника

На цьому етапі всі частини програми об’єднуються в один повний сценарій:

1. Користувач вимовляє wake word **Hey Jarvis**.
2. Система записує аудіо та перевіряє wake word.
3. Якщо wake word розпізнано, система записує голосову команду.
4. Команда перетворюється на текст.
5. Програма виконує відповідну дію.
6. Система озвучує результат.

In [ ]:
print("Скажіть ключову фразу: Hey Jarvis")

wake_audio = record_audio(
    seconds=3,
    filename="wake.wav"
)

wake_detected = check_wake_word(
    audio_path=wake_audio,
    threshold=0.3
)

if wake_detected:
    await speak("Я слухаю")

    print("Скажіть голосову команду")

    command_audio = record_audio(
        seconds=5,
        filename="command.wav"
    )

    command_text = recognize_speech(command_audio)

    response = execute_command(command_text)

    print("Відповідь системи:", response)

    await speak(response)

else:
    response = "Ключове слово не розпізнано"
    print(response)
    await speak(response)

# Висновок

У результаті було створено прототип голосового помічника у середовищі Google Colab.

Розроблена система демонструє базовий принцип роботи голосового керування:

- активація за ключовою фразою;
- розпізнавання голосової команди;
- виконання простої дії;
- озвучення відповіді.
